<a id="top"></a>

# Demo: the same loop, driving a real model

```yaml
title: "Demo: the same loop, driving a real model"
keywords: agent loop, run_loop, live model, langchain, bind_tools, tool_calls, ToolMessage, model-agnostic, multi-step, natural termination, max_steps, sonnet, teacher demo
estimated duration: 12
```

> **Lesson:** L10. Teacher demo — the live counterpart to [L1003](L1003_lecture.ipynb). Makes **live** Claude calls when `ANTHROPIC_API_KEY` is set (copy `.env.example` to `.env`); with no key it prints a skip line so a keyless run still passes. A real model **varies**: dry-run before class. Clear outputs before committing.
>
> The point to land: the loop you built on a stub is the **same loop** that drives a real model. `run_loop` is byte-for-byte the offline version — a LangChain chat model and the `FakeModel` share one interface (`.bind_tools()` → `.invoke()` → `AIMessage.tool_calls`). Only the model object changes; the model chains two-plus tool calls and terminates naturally.

## Contents

- [1. Setup — a real model and the tools](#1-setup--a-real-model-and-the-tools)
- [2. dispatch — unchanged from the stub demo](#2-dispatch--unchanged-from-the-stub-demo)
- [3. run_loop — the same loop, driving a real model](#3-run_loop--the-same-loop-driving-a-real-model)
- [4. Run it — a task that forces two tool calls](#4-run-it--a-task-that-forces-two-tool-calls)
- [5. Takeaways](#5-takeaways)

## 1. Setup — a real model and the tools

A **LangChain chat model** (`init_chat_model("anthropic:…")`, key via the config seam) and the same `calculator` + `lookup` tools. Note what's **gone**: the hand-written JSON tool schemas. `model.bind_tools([calculator, lookup])` infers each tool's schema from its **type hints and docstring**, so the typed function *is* the schema. Swap the `"anthropic:…"` prefix for `"openai:…"` and the identical loop drives a different provider. Anchor model: **Claude Sonnet 4.6**. Live calls are gated on `LIVE`, so a keyless run still passes.

In [ ]:
from collections.abc import Callable
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.messages.tool import ToolCall

from fluffy_potato_curriculum.common.config import get_settings

MODEL = "anthropic:claude-sonnet-4-6"
LIVE = bool(get_settings().anthropic_api_key)


def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the exact result.

    Use for any arithmetic, e.g. '17*17 - 1'. Raises ValueError on non-arithmetic input.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the whitelist above blocks names/attributes/calls.
    return str(eval(expression))


POPULATIONS = {
    "tokyo": "37000000",
    "lagos": "15000000",
    "paris": "11000000",
}


def lookup(city: str) -> str:
    """Look up a city's population from a small internal table.

    Takes a city name, e.g. 'Tokyo'. Raises KeyError if the city is not on file.
    """
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# The dispatch table: tool NAME -> the Python function that runs it. Passed to
# model.bind_tools([...]), these SAME typed functions ARE the schema the model sees:
# LangChain infers each tool's JSON schema from its type hints + docstring, so there
# is no separate hand-written schema to maintain (contrast the old raw SDK).
TOOLS: dict[str, Callable[..., str]] = {"calculator": calculator, "lookup": lookup}

print("live model:", MODEL if LIVE else "(no ANTHROPIC_API_KEY set — the run cell will skip)")

[↑ Back to top](#top)

## 2. dispatch — unchanged from the stub demo

The exact `dispatch` from [L1003](L1003_lecture.ipynb): look up the tool, run it, and turn any exception into a `ToolMessage` with `status="error"`. Loop-level failure handling does not care whether the model is real or fake.

In [ ]:
def dispatch(tools: dict[str, Callable[..., str]], call: ToolCall) -> ToolMessage:
    """Run one requested tool and return a ToolMessage carrying the result.

    On success: a ToolMessage (status="success") with the tool's output. On ANY
    failure (unknown tool name, or the tool raised): a ToolMessage with
    status="error" and a SHORT message (repr(exc)) -- never a traceback. The loop
    keeps going; the model decides what to do next.
    """
    fn = tools.get(call["name"])
    if fn is None:
        return ToolMessage(
            content=f"no such tool {call['name']!r}",
            tool_call_id=call["id"],
            status="error",
        )
    try:
        output = fn(**call["args"])
        return ToolMessage(content=output, tool_call_id=call["id"], status="success")
    except Exception as exc:
        # repr(exc) is a class name + one line -- short, descriptive, cheap.
        return ToolMessage(content=repr(exc), tool_call_id=call["id"], status="error")

[↑ Back to top](#top)

## 3. run_loop — the same loop, driving a real model

The same control flow as the stub demo, **unchanged**. `model.bind_tools(...)` then `.invoke(messages)` → an `AIMessage`; read `.tool_calls`; append a `ToolMessage` per call before the next call. Because the loop talks to the LangChain interface, a real model and the `FakeModel` are byte-for-byte interchangeable — the code below is the offline loop with nothing added.

In [ ]:
@dataclass
class RunResult:
    """The answer, the number of model calls, and WHY the loop stopped."""

    final_text: str
    iterations: int
    termination: str


def run_loop(model: object, user_msg: str, max_steps: int) -> RunResult:
    """A model -> tool -> model loop against a real LangChain chat model."""
    bound = model.bind_tools(list(TOOLS.values()))
    messages: list[BaseMessage] = [HumanMessage(content=user_msg)]
    for step in range(1, max_steps + 1):
        reply = bound.invoke(messages)
        if not reply.tool_calls:
            print(f"[step {step}] natural termination")
            return RunResult(reply.text, step, "natural")
        messages.append(reply)
        for call in reply.tool_calls:
            print(f"[step {step}] tool call: {call['name']}({call['args']})")
            messages.append(dispatch(TOOLS, call))
    print(f"[stop] max_steps={max_steps} hit")
    return RunResult("", max_steps, "max_steps")

[↑ Back to top](#top)

## 4. Run it — a task that forces two tool calls

A prompt designed to force a **chain**: compute something with `calculator`, then use that result to `lookup` a city's population, then answer. Watch the per-step prints show two-plus tool calls before natural termination.

In [ ]:
task = (
    "Use the calculator to compute 17 squared minus 1, then look up the population of the "
    "city Tokyo, and tell me both numbers. Use the tools; do not answer from memory."
)
if LIVE:
    model = init_chat_model(MODEL, api_key=get_settings().anthropic_api_key)
    result = run_loop(model, task, max_steps=8)
    print()
    print("final answer:\n", result.final_text)
    print("\niterations :", result.iterations)
    print("termination:", result.termination)
else:
    print("No ANTHROPIC_API_KEY set — skipping the live run.")
    print("Set it in your environment or repo-root .env to drive a real model.")

[↑ Back to top](#top)

## 5. Takeaways

- The loop is **the same code** as the stub demo — only the model object changed (a real `ChatAnthropic` in place of the `FakeModel`). That interchangeability is why we taught the control flow offline first.
- **Model-agnostic by construction.** Swap `"anthropic:…"` for `"openai:…"` in `init_chat_model` and the identical loop drives a different provider — the tools, `run_loop`, and `dispatch` don't change.
- A real model **varies**: the exact tool order and iteration count differ run to run. The loop's job is to be correct regardless — the `max_steps` cap is there precisely because you can't predict the model.
- The per-step `print()` is a **minimum-viable trace** (iteration, tool calls, termination cause). [L12](../L12/L1201_intro.md) replaces it with structured tracing on *this exact loop* — keep this code accessible.
- Don't reach for a framework yet: this small loop is the spine LangGraph (L04) will later wrap. Hand-rolling it once is what makes the framework legible.

[↑ Back to top](#top)